In [ ]:
# Day 14 - LangChain for Easy LLM Applications
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from transformers import pipeline
import torch

In [ ]:
# 1. Create Knowledge Base
knowledge = """
Addis Ababa is the capital of Ethiopia.
Ethiopia is the only African country never colonized.
Injera is the national dish made from teff.
Lalibela is famous for its rock-hewn churches.
Ethiopia is the birthplace of coffee.
AI ecosystem is growing fast in Ethiopia, especially in Addis Ababa.
Hayat software enginnier student at addis abab university.
"""

with open("ethiopia_knowledge.txt", "w") as f:
    f.write(knowledge)

# Load and split documents
loader = TextLoader("ethiopia_knowledge.txt")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)
print(f"Created {len(texts)} document chunks")

In [ ]:
# 2. Create Vector Store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(texts, embeddings)
print("Vector store created!")

In [ ]:
# 4. Create RAG Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
)

# Test
response = qa_chain.run("What is special about Ethiopia?")
print(response)

In [ ]:
# 5. Simple Chat Function
def ask_question(question):
    result = qa_chain({"query": question})
    print("Question:", question)
    print("Answer:", result['result'])
    return result['result']

ask_question("Tell me about Ethiopian food and culture.")